In [ ]:
#08/01/2025

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

# %load_ext autoreload
# %autoreload 2

In [2]:

from sian.data import final_save_header
from sian.data import final_save_dataset
from sian.data import final_save_labels
from sian.utils import gettimestamp

from sian.synth import Discrete_HierarchicalNoiseModel_InputDataGenerator
from sian.synth import DiscreteToContinuous_PurifiedCoefficients3D_OutputDataFunction


base_save_dataset_path = "data/"
base_save_dataset_path = "data2/"
# base_save_dataset_path = "../../../data/"
datetimestr = gettimestamp()

In [3]:
TRN_N = 1000*1000
TST_N = 10*1000
N = TRN_N + TST_N

# D = 10
# I_k = 5

# max_tuples_per_band = {
#     1 : 4,
#     2 : 16,
#     3 : 64,
# }
# max_tuples_per_band = { #TESTING
#     1 : 1,
#     2 : 1,
#     3 : 1,
# }
# band_strengths = {
#     1 : 1/3,
#     2 : 1/3,
#     3 : 1/3,
# }


# anova_purification_type = "marginal_ANOVA"
# # anova_purification_type = "conditional_Sobol_ANOVA"
# # anova_purification_type = "conditional_Hooker_ANOVA"

In [4]:
# Ds = [10,20,30]
# Ds = [8, 10, 12, 15] #since Rahil already chose to explore this aspect, I will focus on the other critical parts
Ds = [10]
list_of_max_tuples_per_band = []
for s in [3,4,5]:
    max_dict_v1 = {}
    max_dict_v2 = {}
    for k in range(1,3+1):
        max_dict_v1[k] = s**k
        max_dict_v2[k] = s
    list_of_max_tuples_per_band.append(max_dict_v1)
    list_of_max_tuples_per_band.append(max_dict_v2)
all_I_ks = [4, 5, 6]
seeds=list(range(5))
seeds=list(range(1))

print('list_of_max_tuples_per_band')
print(list_of_max_tuples_per_band)


band_strengths = {
    1 : 1/3,
    2 : 1/3,
    3 : 1/3,
}

anova_purification_type = "marginal_ANOVA"
# anova_purification_type = "conditional_Sobol_ANOVA"
# anova_purification_type = "conditional_Hooker_ANOVA"

list_of_max_tuples_per_band
[{1: 3, 2: 9, 3: 27}, {1: 3, 2: 3, 3: 3}, {1: 4, 2: 16, 3: 64}, {1: 4, 2: 4, 3: 4}, {1: 5, 2: 25, 3: 125}, {1: 5, 2: 5, 3: 5}]


In [5]:
# seed = 0 

# if True:
#     np.random.seed(seed)
#     band_sparsity_str = "s" + ".".join([str(max_tuples_per_band[k+1]) for k in range(3)])
#     synth_dataset_id = f"SYNTH_simple_discrete_synthwave_v11_D{D}_Ik{I_k}_{band_sparsity_str}_seed{seed}"
#     preproc_owner = None
#     print(synth_dataset_id)
#     save_dataset_path = base_save_dataset_path + synth_dataset_id + "/"


#     # dgp_x = Discrete_HierarchicalNoiseModel_InputDataGenerator(I_k, eps=0.0, sources=D, branching=0, depth=0)
#     dgp_x = Discrete_HierarchicalNoiseModel_InputDataGenerator(I_k, eps=0.50, sources=1, branching=D, depth=1)
#     dgp_y = DiscreteToContinuous_PurifiedCoefficients3D_OutputDataFunction(D, I_k, max_tuples_per_band, band_strengths, anova_purification_type, dgp_x)


#     X_dict = dgp_x.generate_samples(N)
#     X_onehot = np.concatenate([X_dict[d] for d in range(D)],axis=1)
#     y = dgp_y.generate_output_samples(X_onehot)

#     pass

In [6]:
SAVING = False
SAVING = True

def convert_to_saveable_dataset(fullX, fullY, D, I_k, fake_header_dict):
    pass

    readable_labels = {}
    full_readable_labels = {}
    startdim = 0
    for d in range(-1,D):
        full_info_d = {
            "label" : "X"+str(d+1),
            "startdim" : d*I_k,
            "numdims" : I_k,
            "count" : I_k,
            "encoding" : "disc.onehot",
        }
        if d==-1:
            full_info_d = {
                "label" : "Y",
                "startdim" : 0,
                "numdims" : 1,
                "encoding" : "cts.raw",
            }
        readable_labels[d] = full_info_d["label"]
        full_readable_labels[d] = full_info_d
    full_readable_labels["task_type"] = "regression"
    full_readable_labels["D0"] = D
    full_readable_labels["D"] = D


    # fake_header_dict = {
    #     "Preprocessed Datetime" : datetimestr, 
    #     "dataset_id" : synth_dataset_id,
    #     "preproc_owner" : preproc_owner,

    #     #"load_dataset_path" : load_dataset_path,
    #     #"save_dataset_path" : save_dataset_path,
    # }

    trainval_portion, is_test_split_shuffled, shuffle_test_split_seed = None, False, None
    if trainval_portion is None:
        trainval_portion = 0.80
        trainval_portion = TRN_N/N
        assert int(trainval_portion*N)==TRN_N
    fake_header_dict["is_test_split_shuffled"]  = is_test_split_shuffled
    fake_header_dict["shuffle_test_split_seed"] = shuffle_test_split_seed
    fake_header_dict["trainval_portion"]        = trainval_portion

    # fake_header_dict["datagen_seed"] = seed

    label_stuff = (readable_labels, full_readable_labels)    
    label_dict = {
        "readable_labels" : label_stuff[0],
        "full_readable_labels" : label_stuff[1],
    }

    if SAVING:
        final_save_header(save_dataset_path, fake_header_dict)
        trnvalX,trnvalY,tstX,tstY = final_save_dataset(save_dataset_path, fullX, fullY, 
                                                        trainval_portion, is_test_split_shuffled, shuffle_test_split_seed)
        ### #### #### #### ###
        final_save_labels(save_dataset_path, label_dict)

In [7]:

for D in Ds:
    for I_k in all_I_ks:
        for max_tuples_per_band in list_of_max_tuples_per_band:
            for seed in seeds:
                np.random.seed(seed)
                band_sparsity_str = "s" + ".".join([str(max_tuples_per_band[k+1]) for k in range(3)])
                synth_dataset_id = f"SYNTH_simple_discrete_synthwave_v11_D{D}_Ik{I_k}_{band_sparsity_str}_seed{seed}"
                preproc_owner = None
                print(synth_dataset_id)
                save_dataset_path = base_save_dataset_path + synth_dataset_id + "/"
                fake_header_dict = {
                    "Preprocessed Datetime" : datetimestr, 
                    "dataset_id" : synth_dataset_id,
                    "preproc_owner" : preproc_owner,
                    "datagen_seed" : seed,
                }
            
            
                # dgp_x = Discrete_HierarchicalNoiseModel_InputDataGenerator(I_k, eps=0.0, sources=D, branching=0, depth=0)
                dgp_x = Discrete_HierarchicalNoiseModel_InputDataGenerator(I_k, eps=0.50, sources=1, branching=D, depth=1)
                dgp_y = DiscreteToContinuous_PurifiedCoefficients3D_OutputDataFunction(D, I_k, max_tuples_per_band, band_strengths, anova_purification_type, dgp_x)
            
                X_dict = dgp_x.generate_samples(N)
                X_onehot = np.concatenate([X_dict[d] for d in range(D)],axis=1)
                y = dgp_y.generate_output_samples(X_onehot)
            
                pass

                convert_to_saveable_dataset(X_onehot, y,     D, I_k, fake_header_dict)


SYNTH_simple_discrete_synthwave_v11_D10_Ik4_s3.9.27_seed0
X_onehot (1000000, 40)
	 (2,)
	 (8,)
	 (4,)
	 (8, 9)
	 (4, 8)
	 (6, 7)
	 (3, 6)
	 (1, 4)
	 (0, 3)
	 (4, 7)
	 (7, 8)
	 (2, 7)
	 (0, 1, 9)
	 (1, 2, 7)
	 (3, 6, 7)
	 (1, 3, 6)
	 (4, 7, 9)
	 (6, 7, 9)
	 (1, 6, 8)
	 (1, 5, 7)
	 (0, 7, 8)
	 (3, 8, 9)
	 (5, 6, 9)
	 (0, 5, 9)
	 (1, 6, 9)
	 (0, 5, 8)
	 (1, 7, 9)
	 (1, 2, 4)
	 (1, 3, 5)
	 (0, 4, 9)
	 (3, 5, 8)
	 (0, 5, 6)
	 (1, 7, 8)
	 (1, 5, 6)
	 (3, 4, 5)
	 (3, 4, 7)
	 (1, 8, 9)
	 (0, 2, 3)
	 (0, 4, 8)


full_y var 1.1338178415901123
SYNTH_simple_discrete_synthwave_v11_D10_Ik4_s3.3.3_seed0
X_onehot (1000000, 40)
	 (2,)
	 (8,)
	 (4,)
	 (8, 9)
	 (4, 8)
	 (6, 7)
	 (0, 1, 9)
	 (1, 2, 7)
	 (3, 6, 7)


full_y var 0.9458380820158659
SYNTH_simple_discrete_synthwave_v11_D10_Ik4_s4.16.64_seed0
X_onehot (1000000, 40)
	 (2,)
	 (8,)
	 (4,)
	 (9,)
	 (8, 9)
	 (4, 8)
	 (6, 7)
	 (3, 6)
	 (1, 4)
	 (0, 3)
	 (4, 7)
	 (7, 8)
	 (2, 7)
	 (4, 5)
	 (0, 5)
	 (1, 3)
	 (3, 8)
	 (2, 8)
	 (4, 6)
	 (5, 9)
	 (0, 1, 9)

KeyboardInterrupt: 